In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/msc-deepfake"
LOCAL_HF_CACHE = "/content/hf_cache"

os.environ["HF_HOME"] = LOCAL_HF_CACHE
os.environ["HF_HUB_CACHE"] = LOCAL_HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = LOCAL_HF_CACHE
os.makedirs(LOCAL_HF_CACHE, exist_ok=True)

OUT_FOLDER = f"{DRIVE_ROOT}/generated_videos/ltx"
PROMPTS_FILE = f"{DRIVE_ROOT}/batch_prompts/week2_prompts_v1.txt"

print(f"DRIVE_ROOT: {DRIVE_ROOT}")
print(f"OUT_FOLDER: {OUT_FOLDER}")

DRIVE_ROOT: /content/drive/MyDrive/msc-deepfake
OUT_FOLDER: /content/drive/MyDrive/msc-deepfake/generated_videos/ltx


In [4]:
!pip install -q --upgrade diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg
print("Installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 156.5 MB/s eta 0:00:00
Installed.


In [5]:
import torch
from diffusers import LTXPipeline

print("Loading LTX-Video...")
pipe = LTXPipeline.from_pretrained(
    "Lightricks/LTX-Video",
    torch_dtype=torch.bfloat16,
)
pipe.to("cuda")
print("Model loaded.")
print(f"GPU free: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading LTX-Video...


model_index.json:   0%|          | 0.00/412 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Model loaded.
GPU free: 8.79 GB


In [6]:
import shutil

FAILED_FOLDER = f"{DRIVE_ROOT}/generated_videos/ltx_v1_FAILED"
os.makedirs(FAILED_FOLDER, exist_ok=True)

moved = 0
for v in os.listdir(OUT_FOLDER):
    if v.endswith(".mp4"):
        shutil.move(f"{OUT_FOLDER}/{v}", f"{FAILED_FOLDER}/{v}")
        moved += 1

print(f"Moved {moved} videos to FAILED folder")
print(f"OUT_FOLDER now: {os.listdir(OUT_FOLDER)}")

Moved 40 videos to FAILED folder
OUT_FOLDER now: []


In [7]:
import datetime
from diffusers.utils import export_to_video

CINEMATIC_SUFFIX = ", natural lighting, cinematic quality, high detail, realistic"
NEGATIVE_PROMPT = "blurry, low quality, distorted, unnatural, deformed, static, motion smear, garbled, artefacts, worst quality"

def generate_and_save(prompt_id, category, prompt_text, out_folder, seed=42):
    full_prompt = prompt_text + CINEMATIC_SUFFIX
    print(f"\n[{prompt_id}] {category}: {prompt_text[:60]}...")
    generator = torch.Generator(device="cuda").manual_seed(seed)
    video = pipe(
        prompt=full_prompt,
        negative_prompt=NEGATIVE_PROMPT,
        width=704,
        height=480,
        num_frames=73,
        num_inference_steps=30,
        guidance_scale=3.0,
        generator=generator,
    ).frames[0]
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{prompt_id}_ltx_{timestamp}.mp4"
    out_path = f"{out_folder}/{filename}"
    export_to_video(video, out_path, fps=24)
    print(f"Saved: {out_path}")
    return out_path

print("Function defined with Week 1 parameters.")

Function defined with Week 1 parameters.


In [8]:
with open(PROMPTS_FILE) as f:
    lines = [line.strip() for line in f
             if line.strip() and not line.strip().startswith("#") and "|" in line]

test_line = lines[0]
parts = test_line.split("|", 2)
prompt_id, category, prompt_text = parts

print(f"Testing settings on: {prompt_id}")
generate_and_save(prompt_id, category, prompt_text, OUT_FOLDER)

Testing settings on: w2_001

[w2_001] portrait: A woman in her thirties smiling and speaking directly to the...


  0%|          | 0/30 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/ltx/w2_001_ltx_20260719_105436.mp4


'/content/drive/MyDrive/msc-deepfake/generated_videos/ltx/w2_001_ltx_20260719_105436.mp4'

In [9]:
existing_ids = set()
for v in os.listdir(OUT_FOLDER):
    if v.endswith(".mp4"):
        prompt_id = v.split("_ltx_")[0]
        existing_ids.add(prompt_id)

remaining = [line for line in lines if line.split("|")[0] not in existing_ids]

print(f"Already done: {len(existing_ids)}")
print(f"Remaining: {len(remaining)}")
print("=" * 60)

successes = []
failures = []

for line in remaining:
    parts = line.split("|", 2)
    if len(parts) != 3:
        continue
    prompt_id, category, prompt_text = parts
    try:
        generate_and_save(prompt_id, category, prompt_text, OUT_FOLDER)
        successes.append(prompt_id)
    except Exception as e:
        print(f"FAILED on {prompt_id}: {e}")
        failures.append((prompt_id, str(e)))

print("\n" + "=" * 60)
print(f"Done. Success: {len(successes)}, Failed: {len(failures)}")
if failures:
    print(f"Failed prompts: {failures}")

Already done: 1
Remaining: 39

[w2_002] portrait: An older man laughing while telling a story, cafe interior, ...


  0%|          | 0/30 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/ltx/w2_002_ltx_20260719_105611.mp4

[w2_003] portrait: A young man with glasses reading a book, close-up on his fac...


  0%|          | 0/30 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/ltx/w2_003_ltx_20260719_105639.mp4

[w2_004] portrait: A woman with curly hair looking thoughtfully out a rainy win...


...

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/ltx/w2_038_ltx_20260719_111311.mp4

[w2_039] edge_case: A crowded market at night with many people walking, string l...


  0%|          | 0/30 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/ltx/w2_039_ltx_20260719_111340.mp4

[w2_040] edge_case: A person walking their dog past a shop window, both dog and ...


  0%|          | 0/30 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/ltx/w2_040_ltx_20260719_111408.mp4

Done. Success: 39, Failed: 0
